In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen3-0.6B"
model_path = "/Users/entingwu/PycharmProjects/Finetune/model/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# modelscope

/Users/entingwu/miniconda3/envs/finetune_qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W1107 20:43:52.529000 810 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 311/311 [00:02<00:00, 139.35it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [2]:
messages = [
  {
    "role": "user", 
    "content": "深圳的天气怎么样"
  }
]

tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "查询指定城市的天气情况",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {
            "type": "string",
            "description": "需要查询天气的城市名称"
          }
        },
        "required": ["city"]
      }
    }
  }
]

text = tokenizer.apply_chat_template(
  messages, 
  tools=tools,
  tokenize=False, 
  add_generation_prompt=True)
print(text)

<|im_start|>system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "查询指定城市的天气情况", "parameters": {"type": "object", "properties": {"city": {"type": "string", "description": "需要查询天气的城市名称"}}, "required": ["city"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
深圳的天气怎么样<|im_end|>
<|im_start|>assistant



In [3]:
result = tokenizer(text, return_tensors="pt").to(model.device)
print(result)
input_length = result["input_ids"].shape[1]
print(result["input_ids"].shape)

{'input_ids': tensor([[151644,   8948,    198,      2,  13852,    271,   2610,   1231,   1618,
            825,    476,    803,   5746,    311,   7789,    448,    279,   1196,
           3239,    382,   2610,    525,   3897,    448,    729,  32628,   2878,
            366,  15918,   1472,  15918,     29,  11874,   9492,    510,     27,
          15918,    397,   4913,   1313,    788,    330,   1688,    497,    330,
           1688,    788,   5212,    606,    788,    330,    455,  69364,    497,
            330,   4684,    788,    330,  51154, 105146, 107581, 104307,  99559,
            497,    330,  13786,    788,   5212,   1313,    788,    330,   1700,
            497,    330,  13193,    788,   5212,   8926,    788,   5212,   1313,
            788,    330,    917,    497,    330,   4684,    788,    330,  85106,
          51154, 104307, 105961,  29991,   9207,   2137,    330,   6279,    788,
           4383,   8926,   1341,   3417,    532,    522,  15918,   1339,   2461,
           181

In [4]:
output_result = model.generate(
  **result,
  max_new_tokens=500,
  do_sample=True,
  temperature=0.7,
  top_k=20,
  top_p=0.9
)
print(output_result)
print(output_result.shape)

tensor([[151644,   8948,    198,      2,  13852,    271,   2610,   1231,   1618,
            825,    476,    803,   5746,    311,   7789,    448,    279,   1196,
           3239,    382,   2610,    525,   3897,    448,    729,  32628,   2878,
            366,  15918,   1472,  15918,     29,  11874,   9492,    510,     27,
          15918,    397,   4913,   1313,    788,    330,   1688,    497,    330,
           1688,    788,   5212,    606,    788,    330,    455,  69364,    497,
            330,   4684,    788,    330,  51154, 105146, 107581, 104307,  99559,
            497,    330,  13786,    788,   5212,   1313,    788,    330,   1700,
            497,    330,  13193,    788,   5212,   8926,    788,   5212,   1313,
            788,    330,    917,    497,    330,   4684,    788,    330,  85106,
          51154, 104307, 105961,  29991,   9207,   2137,    330,   6279,    788,
           4383,   8926,   1341,   3417,    532,    522,  15918,   1339,   2461,
           1817,    729,   1

In [5]:
all_text = tokenizer.decode(output_result[0], skip_special_tokens=True)
print(all_text)

system
# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "查询指定城市的天气情况", "parameters": {"type": "object", "properties": {"city": {"type": "string", "description": "需要查询天气的城市名称"}}, "required": ["city"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
user
深圳的天气怎么样
assistant
<think>
好的，用户问的是深圳的天气怎么样。我需要使用提供的工具来查询。工具里有一个get_weather函数，需要参数city。用户提到的地点是深圳，所以应该调用这个函数，并传入“深圳”作为参数。确认一下参数是否正确，城市名称是否正确。然后生成对应的tool_call XML，确保格式正确。不需要其他步骤，直接返回结果即可。
</think>

<tool_call>
{"name": "get_weather", "arguments": {"city": "深圳"}}
</tool_call>


In [6]:
input_length = result["input_ids"].shape[1]
print('input_length', result["input_ids"].shape)

input_length torch.Size([1, 160])


In [7]:
generated_result = output_result[0, input_length:]
print(generated_result)

tensor([151667,    198,  99692,   3837,  20002,  56007, 100146, 101119,   9370,
        104307, 104472,   1773,  35946,  85106,  37029, 103008, 102011,  36407,
         51154,   1773, 102011,  69249, 104133,    455,  69364,  32804,   3837,
         85106,  32665,   8926,   1773,  20002, 104496,   9370, 104766,  20412,
        101119,   3837,  99999,  99730,  47872,  11622,  99487,  32804,  90395,
         41683,  17254,   2073, 101119,    854, 100622,  32665,   1773,  81167,
        100158,  32665,  64471,  88991,   3837,  99490,  29991,  64471,  88991,
          1773, 101889,  43959, 110019,  14172,  13429,  11874,   3837, 103944,
         68805,  88991,   1773, 104689,  92894, 105652,   3837, 101041,  31526,
         59151, 104180,   8997, 151668,    271, 151657,    198,   4913,    606,
           788,    330,    455,  69364,    497,    330,  16370,    788,   5212,
          8926,    788,    330, 101119,  95642, 151658, 151645],
       device='mps:0')


In [8]:
generated_text = tokenizer.decode(generated_result, skip_special_tokens=True)
print(generated_text)

<think>
好的，用户问的是深圳的天气怎么样。我需要使用提供的工具来查询。工具里有一个get_weather函数，需要参数city。用户提到的地点是深圳，所以应该调用这个函数，并传入“深圳”作为参数。确认一下参数是否正确，城市名称是否正确。然后生成对应的tool_call XML，确保格式正确。不需要其他步骤，直接返回结果即可。
</think>

<tool_call>
{"name": "get_weather", "arguments": {"city": "深圳"}}
</tool_call>


In [9]:
from transformers import TextIteratorStreamer
from threading import Thread

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

generation_kwargs = {
  **result,
  "max_new_tokens": 512,
  "do_sample": True,
  "temperature": 0.7,
  "top_p": 0.8,
  "top_k": 20,
  "streamer": streamer,
}

thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

print("Assistant: ", end="", flush=True)
for text_piece in streamer:
    print(text_piece, end="", flush=True)

thread.join()

Assistant: <think>
好的，用户问的是深圳的天气怎么样。我需要调用天气查询工具来获取信息。首先，确认工具的参数是城市名称，所以参数应该是“深圳”。然后，生成对应的JSON请求，确保格式正确。检查是否有其他参数需要，但这里只有城市名称，所以没问题。最后，返回工具调用的JSON结果。
</think>

<tool_call>
{"name": "get_weather", "arguments": {"city": "深圳"}}
</tool_call>